# 04A — Modelling readiness

## Objective and scope
This notebook assesses whether the supplied files are ready for the first regression experiment. It does not merge cohorts, alter source data, impute values, drop duplicate or unusual observations, create an accepted SER target, or train predictive models.

**Usable rows** below means *structurally complete candidate rows*: rows with parseable candidate target components and observed candidate predictors for that dataset. This is a data-availability count only. It does not certify clinical validity, remove sentinel candidates, deduplicate observations, or establish the target definition.

In [1]:
from pathlib import Path
import shutil, subprocess, tempfile, re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
if not (ROOT/"DATA.xls").exists(): ROOT=ROOT.parent
AUDIT=ROOT/"outputs"/"audit"; AUDIT.mkdir(parents=True,exist_ok=True)
# Read legacy .xls through a temporary conversion if the xlrd engine is unavailable.
try:
    xls=pd.read_excel(ROOT/"DATA.xls",sheet_name=None)
except ImportError:
    soffice=shutil.which("soffice") or "/Users/omvdangi/.cache/codex-runtimes/codex-primary-runtime/dependencies/bin/override/soffice"
    temp=Path(tempfile.mkdtemp(prefix="myopia_readiness_"))
    subprocess.run([soffice,"--headless","--convert-to","xlsx","--outdir",str(temp),str(ROOT/"DATA.xls")],check=True,capture_output=True,text=True)
    xls=pd.read_excel(temp/"DATA.xlsx",sheet_name=None)
data3=pd.read_excel(ROOT/"Data3.xlsx",sheet_name=None)
csv=pd.read_csv(ROOT/"DataSet_main.csv",sep=";",decimal=",",engine="python")


## 1. DATA.xls: provisional primary cohort, 2018 sheet
The 2018 sheet is the provisional primary development dataset because it contains the requested right-eye refraction components and all eight proposed demographic/biological/clinical/environmental candidate predictors in the same sheet. That is a structural rationale only; the clinical target remains unverified. The 2014 sheet is reported separately and is not pooled with 2018.

In [2]:
primary=xls["2018"]
target_cols=["spherical refraction of right eye","cylindrical refraction of right eye"]
predictors=["gender","height","weight","axial length of right eye","time of study at weekdays","time of study at weekend days","time outdoors at weekdays","time outdoors at weekend days"]
print("Exact 2018 columns:")
for c in primary.columns: print(" -",repr(c))
missing_cols=[c for c in target_cols+predictors if c not in primary.columns]
parse={c:pd.to_numeric(primary[c],errors="coerce") for c in target_cols+predictors if c in primary.columns}
complete=pd.concat(parse,axis=1).notna().all(axis=1) if len(parse)==len(target_cols+predictors) else pd.Series(False,index=primary.index)
print(f"\n2018 rows={len(primary):,}; exact duplicate rows retained={primary.duplicated().sum():,}; structurally complete candidate rows={int(complete.sum()):,}; missing required columns={missing_cols}")
print("Missing observations in required columns:",primary[target_cols+predictors].isna().sum().to_dict())
print("Sphere/cylinder exact candidate ranges:",primary[target_cols].agg(["min","max"]).to_dict())
print("Sentinel-like values retained for clinical review:",{c:primary[c].value_counts().reindex([-1,-9,-99,999,9999]).dropna().to_dict() for c in target_cols})
# Record, do not exclude, values outside descriptive IQR fences for follow-up only.
for c in ["height","weight","axial length of right eye"]+target_cols:
    s=pd.to_numeric(primary[c],errors="coerce"); q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    print(f"IQR review flag {c!r}: {int(((s<q1-1.5*iqr)|(s>q3+1.5*iqr)).sum())} rows; flags are not validity judgments and no rows removed.")

Exact 2018 columns:
 - 'gender'
 - 'time of study at weekdays'
 - 'time of study at weekend days'
 - 'time outdoors at weekdays'
 - 'time outdoors at weekend days'
 - 'height'
 - 'weight'
 - 'spherical refraction of right eye'
 - 'cylindrical refraction of right eye'
 - 'axis of right eye'
 - 'axial length of right eye'

2018 rows=3,658; exact duplicate rows retained=0; structurally complete candidate rows=3,658; missing required columns=[]
Missing observations in required columns: {'spherical refraction of right eye': 0, 'cylindrical refraction of right eye': 0, 'gender': 0, 'height': 0, 'weight': 0, 'axial length of right eye': 0, 'time of study at weekdays': 0, 'time of study at weekend days': 0, 'time outdoors at weekdays': 0, 'time outdoors at weekend days': 0}
Sphere/cylinder exact candidate ranges: {'spherical refraction of right eye': {'min': -14.0, 'max': 8.25}, 'cylindrical refraction of right eye': {'min': -5.0, 'max': 0.0}}
Sentinel-like values retained for clinical review:

### Interpretation of 2018 target and predictors
The exact candidate columns are: sphere `spherical refraction of right eye`; cylinder `cylindrical refraction of right eye`; axial length `axial length of right eye`; anthropometrics `height`, `weight`; demographic `gender`; study time `time of study at weekdays`, `time of study at weekend days`; outdoor time `time outdoors at weekdays`, `time outdoors at weekend days`.

The audit reports 3,658 rows, zero duplicate rows, zero missing entries in these ten required columns, sphere values from −14 to 8.25, and cylinder values from −5 to 0. The values −1 and −9 occur in sphere and −1 occurs in cylinder. These may be valid measurements or special codes; the available files cannot resolve that. The range checks and IQR flags are review aids only. No rows are removed. Without refraction protocol documentation, a computable `sphere + cylinder/2` remains provisional and should not yet be used as a supervised learning outcome.

## 2. Separate cohort review: Data3.xlsx
This cohort has bilateral sphere and cylinder candidate fields, anthropometrics, birth/exam year fields, gender, eye axes, and bilateral axial-length candidates. It is not merged with `DATA.xls`. `birthday` and `examination date` appear as years in the file, but their definitions and age convention are undocumented. The literal `/` appears in object-valued clinical fields; its meaning is unknown and it is not recoded here. Fifteen exact duplicate rows are retained. No participant linkage or protocol equivalence is established.

In [3]:
d3=data3["2014-2017"]
d3_target=["spherical refraction of right eye","cylinder refraction of right eye"]
d3_pred=["gender","height","weight","AL of right eye"]
def numeric_series(frame,col):
    return pd.to_numeric(frame[col].astype("string").str.strip().str.replace(",",".",regex=False),errors="coerce")
d3_numeric=d3_target+["height","weight","AL of right eye"]
d3_complete=pd.concat([numeric_series(d3,c) for c in d3_numeric],axis=1).notna().all(axis=1) & d3["gender"].notna()
print("Rows:",len(d3),"exact duplicate rows retained:",int(d3.duplicated().sum()),"right-eye target + possible common predictors structurally complete:",int(d3_complete.sum()))
print("Missing source cells by field:",d3.isna().sum()[d3.isna().sum()>0].to_dict())
for c in d3.columns:
    if d3[c].astype("string").str.strip().eq("/").any(): print("slash-coded observations retained:",repr(c),int(d3[c].astype("string").str.strip().eq("/").sum()))
print("Unresolved: SER convention, sex coding, units, measurement protocol, age derivation, slash-code meaning, and independence from development data.")

Rows: 2024 exact duplicate rows retained: 15 right-eye target + possible common predictors structurally complete: 1877
Missing source cells by field: {'height': 1, 'weight': 1, 'spherical refraction of right eye': 8, 'spherical refraction of left eye': 7, 'cylinder refraction of right eye': 7, 'cylinder refraction of left eye': 8, 'axis of right eye': 48, 'axis of left eye': 60, 'AL of right eye': 1, 'AL of left eye': 1}
slash-coded observations retained: 'cylinder refraction of right eye' 5
slash-coded observations retained: 'cylinder refraction of left eye' 5
slash-coded observations retained: 'axis of right eye' 198
slash-coded observations retained: 'axis of left eye' 198
slash-coded observations retained: 'AL of right eye' 137
slash-coded observations retained: 'AL of left eye' 137
Unresolved: SER convention, sex coding, units, measurement protocol, age derivation, slash-code meaning, and independence from development data.


## 3. Separate cohort review: DataSet_main.csv
The audit detected semicolon separation and comma decimals. The candidate `SE` field parses numerically for 874 of 876 rows; that does not verify its meaning as continuous SER. The 23 columns include `ID`, `NewClass_M`, `Class_M`, `SE`, `Sex`, `Age`, school/family/birth-related abbreviations, and activity/time abbreviations. Without a codebook, their meanings, coding, units, and relationships cannot be safely inferred. `ID` is not a predictor. `NewClass_M` and `Class_M` may encode classes related to refractive status; exclude them from predictors until documented, and in any event do not use outcome-derived categories to predict continuous SER. This dataset remains separate.

In [4]:
print("CSV rows:",len(csv),"exact duplicate rows retained:",int(csv.duplicated().sum()),"fully empty rows:",int(csv.isna().all(axis=1).sum()))
print("Column names:",csv.columns.tolist())
print("SE parseable:",int(pd.to_numeric(csv["SE"],errors="coerce").notna().sum()),"of",len(csv))
print("Rows with SE and all non-ID candidate columns observed:",int(csv.drop(columns=["ID"]).notna().all(axis=1).sum()))
print("Unresolved: SE definition/laterality/units/derivation, all abbreviated variable meanings/codes, participant-ID interpretation, and whether class fields derive from SE.")

CSV rows: 876 exact duplicate rows retained: 1 fully empty rows: 2
Column names: ['ID', 'NewClass_M', 'Class_M', 'SE', 'Sex', 'Age', 'Sch_Level', 'Sch_Location', 'A_SS', 'F_E', 'M_E', 'Gest_Age', 'Delivery', 'BreastF', 'Fam_H', 'Sleep', 'Read_D', 'PA', 'HW', 'OutAct_w', 'OutAct_wkd', 'SP_w', 'SP_wkd']
SE parseable: 874 of 876
Rows with SE and all non-ID candidate columns observed: 874
Unresolved: SE definition/laterality/units/derivation, all abbreviated variable meanings/codes, participant-ID interpretation, and whether class fields derive from SE.


## 4. Target leakage and experiment design
For a right-eye SER target calculated from the DATA.xls 2018 sphere/cylinder pair, including either or both component columns as predictors leaks the target by construction. A direct SER/SE column is the target itself and must not appear on the predictor side. Any refractive class generated from the target is also leakage. Identifiers can create memorization and have no defensible biological predictive meaning. Measurements collected after or as part of the target measurement may also create availability or incorporation bias; timing must be documented.

If the clinical documentation verifies SER and predictor timing/meaning, candidate predictors are the eight listed above. All transformations, imputers, encoders, and scalers must be fitted inside cross-validation/training pipelines. Keep participant-level groups if identifiers reveal repeated records, and reserve the test set for final evaluation. Do not pool 2014 with 2018 or use Data3/CSV as external validation without compatibility evidence.

In [5]:
# Readiness decisions are explicit; checklist entries do not imply model training.
checklist=pd.DataFrame([
 ("Target formula confirmed against clinical protocol and refraction notation",False,"No codebook/protocol supplied"),
 ("Right-eye laterality and measurement conditions confirmed",False,"Column names indicate right eye; protocol unavailable"),
 ("Predictor definitions, units, coding, and prediction-time availability confirmed",False,"Gender/time units and measurement timing undocumented"),
 ("Sentinel-like values adjudicated",False,"-1/-9 are observed; meanings unverified"),
 ("Duplicate/repeated-participant structure understood",False,"No participant IDs in DATA.xls; cross-sheet overlap cannot be checked"),
 ("Development/test split strategy can prevent participant leakage",False,"No identifiers/groups available to verify repeated participants"),
 ("Independent validation compatibility established",False,"Data3 and CSV definitions/protocols unresolved"),
 ("Model training authorized by scientific readiness",False,"Blocked pending items above")],columns=["check","complete","evidence_or_missing_item"])
display(checklist); checklist.to_csv(AUDIT/"modelling_readiness_checklist.csv",index=False)

,check,complete,evidence_or_missing_item
0,Target formula confirmed against clinical prot...,False,No codebook/protocol supplied
1,Right-eye laterality and measurement condition...,False,Column names indicate right eye; protocol unav...
2,"Predictor definitions, units, coding, and pred...",False,Gender/time units and measurement timing undoc...
3,Sentinel-like values adjudicated,False,-1/-9 are observed; meanings unverified
4,Duplicate/repeated-participant structure under...,False,No participant IDs in DATA.xls; cross-sheet ov...
5,Development/test split strategy can prevent pa...,False,No identifiers/groups available to verify repe...
6,Independent validation compatibility established,False,Data3 and CSV definitions/protocols unresolved
7,Model training authorized by scientific readiness,False,Blocked pending items above


## 5. Machine-readable readiness report
`usable_rows` is the count of structurally complete candidate rows using the fields stated in each row. It is not a count of clinically valid target observations. For the CSV, the count is based on a parseable `SE` and observed non-ID columns; because definitions are missing, this count only describes completeness, not an approved predictor set.

In [6]:
report=pd.DataFrame([
 {"dataset":"DATA.xls (2018)","target_candidate":"spherical refraction of right eye + cylindrical refraction of right eye; candidate SER = sphere + cylinder/2","target_status":"Unverified; formula must await refraction convention/protocol confirmation","usable_rows":int(complete.sum()),"predictor_candidates":"gender; height; weight; axial length of right eye; time of study at weekdays; time of study at weekend days; time outdoors at weekdays; time outdoors at weekend days","unresolved_issues":"Sphere/cylinder convention and measurement protocol; units/coding and measurement timing; -1/-9 sentinel meanings; participant linkage/repeated-measure status","modelling_status":"NOT READY — do not train"},
 {"dataset":"Data3.xlsx (2014-2017)","target_candidate":"Right-eye sphere + right-eye cylinder/2; separate left-eye pair also present","target_status":"Unverified; same clinical documentation needed","usable_rows":int(d3_complete.sum()),"predictor_candidates":"gender; height; weight; AL of right eye (candidate common set)","unresolved_issues":"Target convention/protocol; units and gender coding; slash values; age-year definitions; exact duplicates retained; independence from DATA.xls unknown","modelling_status":"NOT READY — separate cohort; no pooling/validation yet"},
 {"dataset":"DataSet_main.csv", "target_candidate":"SE (name-based candidate only)","target_status":"Unverified; 874 values parse numerically but definition/laterality/derivation unknown","usable_rows":int((pd.to_numeric(csv["SE"],errors="coerce").notna() & csv.drop(columns=["ID"]).notna().all(axis=1)).sum()),"predictor_candidates":"No verified predictors; Sex, Age, school/family/activity abbreviations are unverified candidates; ID excluded; NewClass_M/Class_M potential target-derived fields excluded pending documentation","unresolved_issues":"SE meaning, laterality, units, derivation; all code mappings; class-field derivation; ID semantics; cohort protocol","modelling_status":"NOT READY — retain as independent cohort"}])
report.to_csv(AUDIT/"modelling_readiness_report.csv",index=False)
display(report)

,dataset,target_candidate,target_status,usable_rows,predictor_candidates,unresolved_issues,modelling_status
0,DATA.xls (2018),spherical refraction of right eye + cylindrica...,Unverified; formula must await refraction conv...,3658,gender; height; weight; axial length of right ...,Sphere/cylinder convention and measurement pro...,NOT READY — do not train
1,Data3.xlsx (2014-2017),Right-eye sphere + right-eye cylinder/2; separ...,Unverified; same clinical documentation needed,1877,gender; height; weight; AL of right eye (candi...,Target convention/protocol; units and gender c...,NOT READY — separate cohort; no pooling/valida...
2,DataSet_main.csv,SE (name-based candidate only),Unverified; 874 values parse numerically but d...,874,"No verified predictors; Sex, Age, school/famil...","SE meaning, laterality, units, derivation; all...",NOT READY — retain as independent cohort


In [7]:
print("CONCLUSION")
print("Ready to train: NO. No predictive models were run and no model scores were generated.")
print("Still missing: verified sphere/cylinder convention and refraction protocol; definitions, units, coding, and measurement timing for candidate predictors; adjudication of sentinel-like values; and participant/repeated-measure information.")
print("Next step: obtain the source codebook and clinical measurement documentation, resolve these items without deleting records silently, then freeze a documented target/predictor specification and leakage-safe split plan before model training.")

CONCLUSION
Ready to train: NO. No predictive models were run and no model scores were generated.
Still missing: verified sphere/cylinder convention and refraction protocol; definitions, units, coding, and measurement timing for candidate predictors; adjudication of sentinel-like values; and participant/repeated-measure information.
Next step: obtain the source codebook and clinical measurement documentation, resolve these items without deleting records silently, then freeze a documented target/predictor specification and leakage-safe split plan before model training.
